**<h1 align="center">SBERT Baseline Evaluation</h1>**

This notebook evaluates SBERT performance on the validation and test split and compares the off-the-shelf model with the fine-tuned baseline. It assesses how fine-tuning improves CV–job matching by comparing retrieval performance and inspecting ranked predictions. The notebook is executed on Google Colab to leverage GPU acceleration.

# Table of Contents
* [1. Imports & Setup](#chapter1)
* [2. Load Data & Models](#chapter2)
* [3. Validation](#chapter3)
* [4. Test](#chapter4)
* [5. Analysis](#chapter5)

<a class="anchor" id="chapter1"></a>

# 1. Imports & Setup

</a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
import json
import re
from datetime import datetime
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util

In [3]:
BASE = "/content/drive/MyDrive/NOVA IMS/Year 2/Thesis"
if BASE not in sys.path:
    sys.path.insert(0, BASE)

from thesis_utility import (
    build_ir_eval_unique_jobs_df,
    save_json,
    metrics_dict_to_series,
)

In [4]:
# Check if running on GPU
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [5]:
DATA_DIR = f"{BASE}/Data"
OUT_DIR = f"{BASE}/Results"
TIME = datetime.now().strftime("%m%d%Y_%H%M")

In [6]:
MODEL_OFF_SHELF = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_TUNED = f"{BASE}/Models/sbert_baseline_epoch1_bs128_20260113_1453"

<a class="anchor" id="chapter2"></a>

# 2. Load Data & Models

</a>

In [7]:
val_df = pd.read_csv(f"{DATA_DIR}/val.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

In [8]:
print(len(val_df))
print(len(test_df))

88489
88643


In [9]:
model_off_shelf = SentenceTransformer(MODEL_OFF_SHELF)
model_tuned = SentenceTransformer(MODEL_TUNED)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:01<?, ?it/s]

<a class="anchor" id="chapter3"></a>

# 3. Validation

</a>

In [10]:
val_evaluator = build_ir_eval_unique_jobs_df(val_df, name="val-ir")
off_shelf_val_metrics = val_evaluator(model_off_shelf)
tuned_val_metrics = val_evaluator(model_tuned)

In [ ]:
#save_json(off_shelf_val_metrics, f"SBERT_off_shelf_val_metrics_{TIME}.json", out_dir=OUT_DIR)
#save_json(tuned_val_metrics, f"SBERT_tuned_val_metrics_{TIME}.json", out_dir=OUT_DIR)

Saved: /content/drive/MyDrive/NOVA IMS/Year 2/Thesis/Results/SBERT_off_shelf_val_metrics_01202026_1923.json
Saved: /content/drive/MyDrive/NOVA IMS/Year 2/Thesis/Results/SBERT_tuned_val_metrics_01202026_1923.json


'/content/drive/MyDrive/NOVA IMS/Year 2/Thesis/Results/SBERT_tuned_val_metrics_01202026_1923.json'

<a class="anchor" id="chapter4"></a>

# 4. Test

</a>

In [11]:
test_evaluator = build_ir_eval_unique_jobs_df(test_df, name="test-ir")
off_shelf_test_metrics = test_evaluator(model_off_shelf)
tuned_test_metrics = test_evaluator(model_tuned)

In [ ]:
#save_json(off_shelf_test_metrics, f"SBERT_off_shelf_test_metrics_{TIME}.json", out_dir=OUT_DIR)
#save_json(tuned_test_metrics, f"SBERT_tuned_test_metrics_{TIME}.json", out_dir=OUT_DIR)

Saved: /content/drive/MyDrive/NOVA IMS/Year 2/Thesis/Results/SBERT_off_shelf_test_metrics_01202026_1923.json
Saved: /content/drive/MyDrive/NOVA IMS/Year 2/Thesis/Results/SBERT_tuned_test_metrics_01202026_1923.json


'/content/drive/MyDrive/NOVA IMS/Year 2/Thesis/Results/SBERT_tuned_test_metrics_01202026_1923.json'

<a class="anchor" id="chapter5"></a>

# 5. Analysis
</a>

In [12]:
METRICS_TO_KEEP = ["recall@1", "recall@5", "recall@10", "recall@20", "mrr@10", "ndcg@10"]

In [13]:
table = pd.DataFrame({
    "SBERT off shelf val": metrics_dict_to_series(off_shelf_val_metrics),
    "SBERT tuned val": metrics_dict_to_series(tuned_val_metrics),
    "SBERT off shelf test": metrics_dict_to_series(off_shelf_test_metrics),
    "SBERT tuned test": metrics_dict_to_series(tuned_test_metrics),
})
table = table.loc[METRICS_TO_KEEP]
table

,SBERT off shelf val,SBERT tuned val,SBERT off shelf test,SBERT tuned test
recall@1,0.023359,0.020861,0.023589,0.021220
recall@5,0.075230,0.071636,0.075516,0.072459
recall@10,0.109776,0.108307,0.108514,0.109574
recall@20,0.155579,0.156743,0.155083,0.157959
mrr@10,0.045793,0.043075,0.045936,0.043642
ndcg@10,0.060749,0.058274,0.060595,0.059005
